This notebook trains a simple linear Ridge regression model on the running dataset stored in the SQLite database.

Features used: distance (meters), age, year, sex (0/1). Target: time in seconds.

In [2]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "/Users/darraghdonnelly/dev/Database/recovered.db"
TEST_SIZE = 0.20

# connect to db
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        f"SELECT src, sex, age, distance_m, time_s FROM concat_results",
        conn,
    )

print(f"rows: {len(df)}")
df.head()

rows: 143442


,src,sex,age,distance_m,time_s
0,CherryBlossom2017,1,21,16093,3217.0
1,CherryBlossom2017,1,22,16093,3232.0
2,CherryBlossom2017,1,31,16093,3276.0
3,CherryBlossom2017,1,33,16093,3285.0
4,CherryBlossom2017,1,35,16093,3288.0


In [3]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# set features and targets
features = ["distance_m", "age", "sex"]
X = df[features]
y = df["time_s"].astype(float)

# split data into train test using sklearn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42
)

# train a simple linear ridge regression model
model = Ridge(alpha=1.0, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# model metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"samples: {len(df)}")
print(f"MAE:  {mae:.2f} s")
print(f"R2:   {r2:.4f}")

# time conversions
distance_to_time = model.coef_[0] * 1000 / 60
age_to_time = model.coef_[1] / 60
sex_to_time = model.coef_[2] / 60

# basic interpretations
print("\nCoefficients (added time per unit of feature):")

print(f"\nPrediction time per added meter: +{distance_to_time:.6f} min/km")
print(f"Per year added to Age: +{age_to_time:.6f} min/year")
print(f"Sex: +{sex_to_time:.6f} min difference (male vs female)")
print(f"intercept: {model.intercept_:.6f}")

samples: 143442
MAE:  2274.79 s
R2:   0.7167

Coefficients (added time per unit of feature):

Prediction time per added meter: +6.820760 min/km
Per year added to Age: +1.080355 min/year
Sex: +28.443940 min difference (male vs female)
intercept: -4094.416616


In [4]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# convert distance and time (target) to log scale for Reigel model, store in dataframes
df["log_distance"] = np.log(df["distance_m"].astype(float))
y_log = np.log(df["time_s"].astype(float))

# get aged squared so nonliner relationship can be represented
df["ageSqrd"] = df["age"] ** 2

# set features and targets for log model
X = df[["log_distance", "age", "ageSqrd", "sex"]]

# split data into train test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=TEST_SIZE, random_state=42
)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

log_pred = model.predict(X_test)

# predict time in secs 
pred_log = model.predict(X_test)

# convert log predictions to seconds by getting the exponential
mae_s = round(mean_absolute_error(np.exp(y_test), np.exp(pred_log)), 2)

print("MAE seconds:", mae_s, "->", round(mae_s/60, 2), "min")
print("coefficients:", dict(zip(X.columns, model.coef_)))
print("intercept:", model.intercept_)

MAE seconds: 2189.45 -> 36.49 min
coefficients: {'log_distance': np.float64(1.0377171327549934), 'age': np.float64(-0.007087284596855199), 'ageSqrd': np.float64(0.00013585988128208014), 'sex': np.float64(0.1255707821557215)}
intercept: -1.377470668200548


In [5]:

# prediction function that reads distance, age and sex - returns time in secs
def predict_time_seconds(distance_m: float, age: float, sex: int) -> float:
    x = pd.DataFrame([{
        "log_distance": np.log(float(distance_m)),
        "age": float(age),
        "ageSqrd": float(age ** 2),
        "sex": float(sex)
    }])
    return float(np.exp(model.predict(x)[0]))

# func to format time into hh:mm:ss
def format_time(seconds: float) -> str:
    s = int(round(seconds))
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h}:{m:02d}:{sec:02d}"

# different prediction examples
Eg1 = format_time(predict_time_seconds(5000, 20, 0))
Eg2 = format_time(predict_time_seconds(16093, 30, 1))
Eg3 = format_time(predict_time_seconds(42195, 30, 0))

young = format_time(predict_time_seconds(5000, 15, 0))
old = format_time(predict_time_seconds(5000, 60, 0))

print("5k male 20:", Eg1)
print("10 mile female 30:", Eg2)
print("Marathon male 30:", Eg3)

5k male 20: 0:26:33
10 mile female 30: 1:40:59
Marathon male 30: 4:02:09


In [6]:
import joblib
from pathlib import Path

# save model as an artifact for use on cold starts
artifact = {
    "kind": "log_ridge",
    "feature_order": ["log_distance", "age", "ageSqrd", "sex"],
    "model": model,
}

out_path = Path("/Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models/base_ridge.joblib")
out_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, out_path)
print("Saved:", out_path)


Saved: /Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models/base_ridge.joblib
